# YOLOv3: verify inference, then train on your own data

This notebook first runs a **pretrained YOLOv3-tiny COCO smoke test** on Darknet's public sample image. It is a quick check of the runtime and detector, **not a test of a custom-trained YOLOv3 model or its accuracy**. The custom full-YOLOv3 workflow below is opt-in.

- Run in Colab (Python 3); a GPU is recommended for training. CPU smoke testing works but training can be very slow.
- Darknet source is pinned. Its build, the model downloads and image data have separate licenses; review them before use. Colab/CUDA may change.
- No Drive mount, custom data or training starts unless you enable it. The smoke test downloads about 35 MB and builds external code.
- Do not run on private images unless you trust the notebook and runtime. Keep validation/test images independent of training. Save checkpoints outside the temporary Colab VM.

In [ ]:
"""Small, dependency-free helpers embedded in the Colab notebook.

Run tests with ``python -m unittest discover -s tests -v``. If this file changes,
update the helper cell in the notebook too; a test checks they stay identical.
"""
from pathlib import Path
import hashlib
import math
import os
import re
import subprocess
import tempfile
import urllib.request


SUPPORTED_IMAGES = {'.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG'}


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def download_checked(url, path, expected_sha256):
    """Download to a temporary file; reject corrupt data without replacing a good file."""
    path = Path(path)
    if path.is_file() and file_sha256(path) == expected_sha256:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix='.download-', dir=path.parent)
    os.close(fd)
    try:
        # Some weight hosts reject urllib's default user agent. Limit the download
        # size and verify its digest before it can replace an existing file.
        request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(request, timeout=30) as response, open(temp, 'wb') as output:
            total = 0
            while chunk := response.read(1024 * 1024):
                total += len(chunk)
                if total > 200 * 1024 * 1024:
                    raise ValueError('Weight download exceeds the 200 MiB safety limit.')
                output.write(chunk)
        if file_sha256(temp) != expected_sha256:
            raise ValueError(f'Checksum mismatch for {url}; download was not used.')
        os.replace(temp, path)
    finally:
        if os.path.exists(temp):
            os.unlink(temp)
    return path


def set_option(section, key, value):
    pattern = re.compile(r'^\s*' + re.escape(key) + r'\s*=.*$', re.MULTILINE)
    section, count = pattern.subn(f'{key}={value}', section)
    if count != 1:
        raise ValueError(f'Expected exactly one active {key} option, found {count}.')
    return section


def make_training_cfg(source, classes, batch=64, subdivisions=16, max_batches=None):
    """Configure the three YOLOv3 heads by section, not by fragile line numbers."""
    if isinstance(classes, bool) or not isinstance(classes, int) or classes < 1:
        raise ValueError('classes must be a positive integer')
    if batch < 1 or subdivisions < 1 or batch % subdivisions:
        raise ValueError('batch must be positive and divisible by subdivisions')
    if max_batches is None:
        max_batches = max(6000, classes * 2000)
    if not isinstance(max_batches, int) or max_batches < 2:
        raise ValueError('max_batches must be an integer of at least 2')
    sections = re.split(r'(?=^\[[^\]]+\]\s*$)', source, flags=re.MULTILINE)
    blocks = [i for i, block in enumerate(sections) if re.match(r'^\[net\]', block)]
    if len(blocks) != 1:
        raise ValueError('Expected exactly one [net] block')
    net = blocks[0]
    for key, value in [('batch', batch), ('subdivisions', subdivisions), ('max_batches', max_batches),
                       ('steps', f'{int(max_batches * 0.8)},{int(max_batches * 0.9)}')]:
        sections[net] = set_option(sections[net], key, value)
    heads = [i for i, block in enumerate(sections) if re.match(r'^\[yolo\]', block)]
    if len(heads) != 3:
        raise ValueError(f'Expected three YOLOv3 heads, found {len(heads)}')
    for index in heads:
        if index == 0 or not re.match(r'^\[convolutional\]', sections[index - 1]):
            raise ValueError('Each YOLO head must follow its prediction convolution')
        mask = re.search(r'^\s*mask\s*=\s*([0-9,\s]+)$', sections[index], re.MULTILINE)
        if not mask:
            raise ValueError('YOLO head is missing its anchor mask')
        anchors = [part.strip() for part in mask.group(1).split(',') if part.strip()]
        if len(anchors) != 3:
            raise ValueError('Expected three anchors per YOLOv3 head')
        sections[index] = set_option(sections[index], 'classes', classes)
        sections[index - 1] = set_option(sections[index - 1], 'filters', (classes + 5) * len(anchors))
    return ''.join(sections)


def checked_names(names):
    if not isinstance(names, (list, tuple)) or not names or any(not isinstance(name, str) or not name.strip() or '\n' in name or '\r' in name for name in names):
        raise ValueError('Supply one non-empty class name per line')
    result = [name.strip() for name in names]
    if len(set(result)) != len(result):
        raise ValueError('Class names must be distinct')
    return result


def darknet_label_path(image):
    """Mirror the pinned Darknet revision's image-to-label path substitutions."""
    path = str(image)
    for split in ('train2017', 'val2017', 'train2014', 'val2014'):
        path = path.replace(f'/images/{split}/', f'/labels/{split}/')
    path = path.replace('/JPEGImages/', '/labels/')
    return Path(path).with_suffix('.txt')


def validate_split(directory, class_count):
    """Require explicit labels (empty is OK for negatives) and valid normalized boxes."""
    root = Path(directory).expanduser().resolve()
    if not root.is_dir():
        raise ValueError(f'Dataset directory does not exist: {root}')
    images = sorted(p.resolve() for p in root.rglob('*') if p.is_file() and p.suffix in SUPPORTED_IMAGES)
    if not images:
        raise ValueError(f'No supported images found in {root}')
    if len(set(images)) != len(images):
        raise ValueError(f'Duplicate image paths found in {root}')
    for image in images:
        if '\n' in str(image) or '\r' in str(image):
            raise ValueError(f'Image path contains a newline: {image!s}')
        label = darknet_label_path(image)
        if not label.is_file():
            raise ValueError(f'Missing label for {image}: expected {label}; use an empty file for a negative image')
        for number, line in enumerate(label.read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            try:
                if len(parts) != 5 or not re.fullmatch(r'\d+', parts[0]):
                    raise ValueError()
                category = int(parts[0])
                x, y, width, height = map(float, parts[1:])
                if category >= class_count or not all(math.isfinite(v) for v in (x, y, width, height)):
                    raise ValueError()
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < width <= 1 and 0 < height <= 1):
                    raise ValueError()
                if x - width / 2 < -1e-6 or y - height / 2 < -1e-6 or x + width / 2 > 1 + 1e-6 or y + height / 2 > 1 + 1e-6:
                    raise ValueError()
            except ValueError:
                raise ValueError(f'Invalid YOLO label at {label}:{number}; expected class_id x_center y_center width height in [0, 1]') from None
    return images


def prepare_dataset(train_dir, valid_dir, names, output_dir, backup_dir):
    """Write lists and data files only after both splits pass preflight checks."""
    names = checked_names(names)
    train = validate_split(train_dir, len(names))
    valid = validate_split(valid_dir, len(names))
    if set(train) & set(valid):
        raise ValueError('Train and validation image paths overlap')
    train_hashes = {file_sha256(path) for path in train}
    if any(file_sha256(path) in train_hashes for path in valid):
        raise ValueError('Identical image bytes occur in both training and validation; use independent splits')
    output = Path(output_dir).expanduser().resolve()
    backup = Path(backup_dir).expanduser().resolve()
    if output == backup:
        raise ValueError('Output and backup directories must be different to protect existing checkpoints')
    # Darknet strips whitespace in .data values, so reject paths it would misread.
    if any(char.isspace() for char in str(output)) or any(char.isspace() for char in str(backup)):
        raise ValueError('Darknet .data paths cannot contain whitespace; choose output and backup paths without spaces')
    output.mkdir(parents=True, exist_ok=True)
    backup.mkdir(parents=True, exist_ok=True)
    (output / 'obj.names').write_text('\n'.join(names) + '\n', encoding='utf-8')
    (output / 'train.txt').write_text('\n'.join(map(str, train)) + '\n', encoding='utf-8')
    (output / 'valid.txt').write_text('\n'.join(map(str, valid)) + '\n', encoding='utf-8')
    data = f'classes={len(names)}\ntrain={output / "train.txt"}\nvalid={output / "valid.txt"}\nnames={output / "obj.names"}\nbackup={backup}\n'
    (output / 'obj.data').write_text(data, encoding='utf-8')
    return train, valid


def run_detection(darknet_dir, data, cfg, weights, image, threshold=0.25):
    """Run real Darknet inference, failing on process errors or missing fresh output."""
    if not 0 < threshold < 1:
        raise ValueError('threshold must be between 0 and 1')
    root = Path(darknet_dir).resolve()
    for name, value in [('data', data), ('cfg', cfg), ('weights', weights), ('image', image)]:
        if not Path(value).is_file():
            raise FileNotFoundError(f'{name} file not found: {value}')
    output = root / 'predictions.jpg'
    output.unlink(missing_ok=True)
    result = subprocess.run([str(root / 'darknet'), 'detector', 'test', str(Path(data).resolve()),
                             str(Path(cfg).resolve()), str(Path(weights).resolve()), str(Path(image).resolve()),
                             '-dont_show', '-thresh', str(threshold)], cwd=root, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True)
    if not output.is_file() or output.stat().st_size == 0:
        raise RuntimeError('Darknet exited without creating predictions.jpg; check the log and runtime.')
    return result.stdout, output


## 1. Build a pinned Darknet revision

The default attempts a GPU build only when both CUDA (`nvcc`) and a working `nvidia-smi` are available; otherwise it builds for CPU. OpenCV and cuDNN are disabled to reduce runtime dependencies. If the GPU build fails, check the Colab CUDA/driver compatibility or switch to CPU for the smoke test. Training on CPU can be extremely slow.

In [ ]:
import shutil

DARKNET_COMMIT = "59596d7880f6504768df41d6daa586f5cb2b932f"
DARKNET = Path("/content/darknet-yolov3-pinned")

def gpu_available():
    return bool(shutil.which("nvcc") and shutil.which("nvidia-smi") and
                subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0)

USE_GPU = gpu_available()  # Set False to force a CPU build.
CUDA_COMPUTE_CAPABILITY = None  # Optional override, e.g. "7.5" for a T4.
if not DARKNET.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/AlexeyAB/darknet.git", str(DARKNET)], check=True)
    subprocess.run(["git", "-C", str(DARKNET), "checkout", "--detach", DARKNET_COMMIT], check=True)
else:
    actual = subprocess.check_output(["git", "-C", str(DARKNET), "rev-parse", "HEAD"], text=True).strip()
    if actual != DARKNET_COMMIT:
        raise RuntimeError(f"Existing Darknet checkout is at {actual}, not the pinned revision. Use a fresh directory.")
    changed = subprocess.check_output(["git", "-C", str(DARKNET), "status", "--porcelain", "--untracked-files=no"], text=True)
    if changed.strip():
        raise RuntimeError("Existing Darknet checkout has modified tracked files; use a fresh directory.")

# Clean when switching CPU/GPU options to avoid reusing incompatible object files.
subprocess.run(["make", "clean"], cwd=DARKNET, check=True)
build = ["make", "-j2", f"GPU={int(USE_GPU)}", "CUDNN=0", "OPENCV=0", "LIBSO=0"]
if USE_GPU:
    capability = CUDA_COMPUTE_CAPABILITY
    if capability is None:
        query = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                               text=True, capture_output=True, check=True)
        capability = query.stdout.splitlines()[0].strip()
    if not re.fullmatch(r"[0-9]+\.[0-9]+", capability):
        raise ValueError("Set CUDA_COMPUTE_CAPABILITY to your GPU's capability, e.g. 7.5 for a T4.")
    arch = capability.replace(".", "")
    build.append(f"ARCH=-gencode=arch=compute_{arch},code=sm_{arch}")
subprocess.run(build, cwd=DARKNET, check=True)
print("Darknet revision:", DARKNET_COMMIT, "| GPU build:", USE_GPU)


## 2. Run an actual pretrained detection smoke test

This uses the pinned Darknet `data/dog.jpg`, `yolov3-tiny.cfg`, COCO class names and the original pretrained **YOLOv3-tiny** weights. The download's SHA-256 is checked before use. It requires the program to exit successfully, create a fresh annotated image and detect `dog` at threshold 0.25. Different runtime behavior can still occur; a pass is not a benchmark or a guarantee of custom-model quality.

In [ ]:
from IPython.display import Image, display

TINY_WEIGHTS = download_checked(
    "https://pjreddie.com/media/files/yolov3-tiny.weights",
    Path("/content/yolov3-tiny.weights"),
    "dccea06f59b781ec1234ddf8d1e94b9519a97f4245748a7d4db75d5b7080a42c",
)
log, annotated = run_detection(DARKNET, DARKNET / "cfg/coco.data", DARKNET / "cfg/yolov3-tiny.cfg",
                               TINY_WEIGHTS, DARKNET / "data/dog.jpg")
print(log[-5000:])
if not re.search(r"(?m)^dog:\s*\d", log):
    raise RuntimeError("Smoke test did not report a dog above threshold; inspect the log and runtime.")
print("Smoke test passed: dog detected. This does not validate your custom model.")
display(Image(filename=str(annotated)))


## 3. Optional: prepare your custom dataset

Enable `ENABLE_CUSTOM` and edit the paths and class names below. If using Drive, run the optional mount cell first. Unzip a **trusted** dataset yourself. The helper expects separate train/validation folders with each image's YOLO `.txt` label alongside it (or Darknet's COCO `images/train2017` → `labels/train2017` or Pascal VOC `JPEGImages` → `labels` paths). An empty `.txt` explicitly marks a negative image. Supported image suffixes are JPG/JPEG/PNG (all-lowercase or all-uppercase).

Each non-empty label line is `class_id x_center y_center width height`, normalized to [0, 1]. Class IDs start at 0 and follow `CLASS_NAMES`. Split by source, scene or subject where appropriate—not just random frames from the same video. Preflight catches identical image bytes across splits, but not near-duplicates or all annotation mistakes. Data/config/backup paths must contain no whitespace because Darknet strips it from `.data` values.

In [ ]:
# Optional: run only when your dataset or checkpoints are on Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
ENABLE_CUSTOM = False  # Set True after preparing your own data.
CLASS_NAMES = []       # Example: ["part", "bolt"] in the same order as label IDs.
TRAIN_DIR = Path("/content/drive/MyDrive/yolov3_dataset/train")
VALID_DIR = Path("/content/drive/MyDrive/yolov3_dataset/valid")
WORK_DIR = Path("/content/yolov3_custom")
BACKUP_DIR = Path("/content/drive/MyDrive/yolov3_backup")

# False by default: running all cells should never start a long training job.
RUN_TRAINING = False
ALLOW_CPU_TRAINING = False
RESUME_WEIGHTS = None  # Or a matching previous yolov3_training_last.weights path.
RUN_CUSTOM_INFERENCE = False
RUN_VALIDATION_MAP = False
CUSTOM_WEIGHTS = BACKUP_DIR / "yolov3_training_last.weights"
CUSTOM_IMAGE = None  # None uses the first validation image; change to your own image path.


In [ ]:
if ENABLE_CUSTOM:
    if BACKUP_DIR.expanduser().resolve().is_relative_to(Path("/content/drive")) and not os.path.ismount("/content/drive"):
        raise RuntimeError("Mount Google Drive first, or choose a non-Drive backup path. Local /content files are temporary.")
    names = checked_names(CLASS_NAMES)
    train_images, valid_images = prepare_dataset(TRAIN_DIR, VALID_DIR, names, WORK_DIR, BACKUP_DIR)
    cfg_text = make_training_cfg((DARKNET / "cfg/yolov3.cfg").read_text(encoding="utf-8"), len(names))
    training_cfg = WORK_DIR / "yolov3_training.cfg"
    training_cfg.write_text(cfg_text, encoding="utf-8")
    # Avoid mixing old checkpoints with a different class list or configuration.
    for source, target in [(WORK_DIR / "obj.names", BACKUP_DIR / "obj.names"),
                           (training_cfg, BACKUP_DIR / "yolov3_training.cfg")]:
        if target.exists() and target.read_bytes() != source.read_bytes():
            raise RuntimeError(f"{target} differs from this run. Choose a new backup folder.")
    for source in [WORK_DIR / "obj.names", training_cfg]:
        target = BACKUP_DIR / source.name
        if source.resolve() != target.resolve():
            shutil.copy2(source, target)
    print(f"Preflight passed: {len(train_images)} training and {len(valid_images)} validation images, {len(names)} classes.")
    print("Configs:", WORK_DIR, "| checkpoints:", BACKUP_DIR)
else:
    print("Custom dataset preparation skipped; set ENABLE_CUSTOM=True after editing the paths and class names.")


## 4. Optional: train full YOLOv3

Set `RUN_TRAINING=True` and rerun the config, preflight and training cells. This downloads about 162 MB of verified Darknet-53 **backbone** weights unless `RESUME_WEIGHTS` points to an existing checkpoint. Backbone weights are not a complete detector. The generated configuration uses `max(6000, 2000 × classes)` iterations; this is a starting point, **not** a guarantee of convergence. Training may take hours and generate large checkpoints. Interruptions may leave no recent checkpoint.

For a quick setup check, use a small non-sensitive dataset and stop only after seeing the first batches and a checkpoint. Keep enough storage and monitor GPU memory; increase subdivisions if needed and use a fresh backup directory if you change the config.

In [ ]:
if ENABLE_CUSTOM and RUN_TRAINING:
    if not USE_GPU and not ALLOW_CPU_TRAINING:
        raise RuntimeError("No GPU build detected. Training on CPU is very slow; set ALLOW_CPU_TRAINING=True only if intentional.")
    if RESUME_WEIGHTS is not None:
        initial = Path(RESUME_WEIGHTS).expanduser().resolve()
        if not initial.is_file():
            raise FileNotFoundError(f"Resume checkpoint not found: {initial}")
    else:
        initial = download_checked("https://pjreddie.com/media/files/darknet53.conv.74",
            Path("/content/darknet53.conv.74"), "2495c2690283e4e0bc2050cbd4660b77a8074e14e9c11150c6412fd63db496a7")
    subprocess.run([str(DARKNET / "darknet"), "detector", "train", str(WORK_DIR / "obj.data"),
                    str(training_cfg), str(initial), "-dont_show"], cwd=DARKNET, check=True)
else:
    print("Training skipped. Set ENABLE_CUSTOM and RUN_TRAINING to True after preflight.")


## 5. Optional: inspect custom detections and validation mAP

Point `CUSTOM_WEIGHTS` at a checkpoint that matches the generated full-YOLOv3 config and class names. Turn on `RUN_CUSTOM_INFERENCE` to inspect one image; check the log **and** image, including cases with no predictions. To compute Darknet's validation mAP, enable `RUN_VALIDATION_MAP`; use a representative labeled validation set and keep a separate untouched test set for final evaluation. A single picture or training loss cannot establish accuracy. These optional cells require real custom weights; they are not exercised by the pretrained smoke test.

In [ ]:
if ENABLE_CUSTOM and RUN_CUSTOM_INFERENCE:
    image = Path(CUSTOM_IMAGE).expanduser().resolve() if CUSTOM_IMAGE is not None else valid_images[0]
    log, annotated = run_detection(DARKNET, WORK_DIR / "obj.data", training_cfg, CUSTOM_WEIGHTS, image)
    print(log[-5000:])
    print("Review the image and log, even when there are no detections:", image)
    display(Image(filename=str(annotated)))
else:
    print("Custom inference skipped.")

if ENABLE_CUSTOM and RUN_VALIDATION_MAP:
    if not Path(CUSTOM_WEIGHTS).is_file():
        raise FileNotFoundError(f"Custom checkpoint not found: {CUSTOM_WEIGHTS}")
    subprocess.run([str(DARKNET / "darknet"), "detector", "map", str(WORK_DIR / "obj.data"),
                    str(training_cfg), str(CUSTOM_WEIGHTS), "-dont_show"], cwd=DARKNET, check=True)
else:
    print("Validation mAP skipped.")
